In [ ]:
import jax
import jax.numpy as jnp

I am trying to speed up a recursion (which is a bottleneck in my code) using `lax.scan`. At the moment, that seems to be slower or similar in speed than the naive for loop implementation. It looks like the recursion function `f()` is being recompiled each time. Is there a way to have the reference to the function saved so it doesn't recompile?

If I take `f()` out of the code block, I get a big speedup, but I'm not sure how to implement that in practice within a function in my larger code.

## Baseline recursion using a for loop

In [ ]:
%%timeit

# Array size
n_recursions = 100
n_elements = 20

# Recursor array
recursor = jnp.ones((n_elements, n_recursions))

# Init array
ary = jnp.zeros((n_elements, n_recursions))

# Set the first two elements of the array before recursion
ary = ary.at[:, 0].set(jax.random.normal(key=jax.random.PRNGKey(54), shape=(n_elements,)))
ary = ary.at[:, 1].set(jax.random.normal(key=jax.random.PRNGKey(34), shape=(n_elements,)))

# Recursive loop; fill subsequent elements of array
for k in jnp.arange(2, n_recursions):
    n = jnp.arange(k - 1)  # Depends on k; variable-size array
    ary = ary.at[:, k].set(jnp.sum(recursor[:, k - n] * ary[:, n], axis=1))

jnp.sum(ary)

283 ms ± 5.88 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Using `jax.lax.scan`

In [ ]:
%%timeit

# Array size
n_recursions = 100
n_elements = 20

# Recursor array
recursor = jnp.ones((n_elements, n_recursions))

# Init array
ary = jnp.zeros((n_elements, n_recursions))

# Set the first two elements of the array before recursion
ary = ary.at[:, 0].set(jax.random.normal(key=jax.random.PRNGKey(54), shape=(n_elements,)))
ary = ary.at[:, 1].set(jax.random.normal(key=jax.random.PRNGKey(34), shape=(n_elements,)))

def f(ary, k):
    n = jnp.arange(n_recursions - 1)  # Does not depend on k; fixed-size array
    ary = ary.at[:, k].set(jnp.cumsum(recursor[:, k - n] * ary[:, n], axis=1)[:, k - 2])
    return ary, k

ary, _ = jax.lax.scan(f, ary, jnp.arange(2, n_recursions + 1))

jnp.sum(ary)

252 ms ± 6.69 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Taking `f()` out of the code block

Here, I can get a big speedup. However, I'm not sure how to include this kind of setup in a larger piece of code, where everything is in a single function. I've tried to include the external variables `n_recursions`, `n_elements`, and `recursor` in the carry argument of `f()` and then unpack them, but haven't gotten this to work.

In [ ]:
# # Array size
# n_recursions = 100
# n_elements = 20

# # Recursor array
# recursor = jnp.ones((n_elements, n_recursions))

# def f(ary, k):
#     n = jnp.arange(n_recursions - 1)  # Does not depend on k; fixed-size array
#     ary = ary.at[:, k].set(jnp.cumsum(recursor[:, k - n] * ary[:, n], axis=1)[:, k - 2])
#     return ary, k

In [ ]:
# %%timeit

# # Init array
# ary = jnp.zeros((n_elements, n_recursions))

# # Set the first two elements of the array before recursion
# ary = ary.at[:, 0].set(jax.random.normal(key=jax.random.PRNGKey(54), shape=(n_elements,)))
# ary = ary.at[:, 1].set(jax.random.normal(key=jax.random.PRNGKey(34), shape=(n_elements,)))
    
# ary, _ = jax.lax.scan(f, ary, jnp.arange(2, n_recursions + 1))

# jnp.sum(ary)